# Piloto Text2KG — Qwen2.5-7B-Instruct-AWQ

**Configuração obrigatória antes de rodar:**
- Accelerator: GPU T4 x2 (Settings → Accelerator)
- Internet: On (Settings → Internet)
- Secrets adicionados (Settings → Secrets):
  - `GITHUB_TOKEN` — token com acesso read ao repositório privado
  - `HF_TOKEN` — token HuggingFace com acesso ao modelo AWQ

**Tempo estimado:** ~3h para 200 frases × 2 ontologias × 3 condições + indução C2.

In [ ]:
# Célula 1 — Variáveis de ambiente e caminhos
import os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN     = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

REPO_URL   = f"https://{GITHUB_TOKEN}@github.com/fmsoares/llm-kg-ontology.git"  # ajuste se necessário
BENCH_URL  = "https://github.com/nandana/Text2KGBench.git"

WORK_DIR   = "/kaggle/working"
REPO_DIR   = f"{WORK_DIR}/llm-kg-ontology"
BENCH_DIR  = f"{WORK_DIR}/Text2KGBench"
RESULT_DIR = f"{REPO_DIR}/resultados/piloto"
C2_DIR     = f"{REPO_DIR}/configs/c2"

print(f"REPO_DIR:   {REPO_DIR}")
print(f"BENCH_DIR:  {BENCH_DIR}")
print(f"RESULT_DIR: {RESULT_DIR}")

In [ ]:
# Célula 2 — Clonar repositórios
import subprocess

def run(cmd, **kw):
    """Executa comando e imprime saída."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Comando falhou (rc={result.returncode}): {cmd}")
    return result

if not os.path.exists(REPO_DIR):
    run(f"git clone --depth=1 {REPO_URL} {REPO_DIR}")
else:
    print("Repositório já existe — atualizando...")
    run(f"git -C {REPO_DIR} pull")

if not os.path.exists(BENCH_DIR):
    run(f"git clone --depth=1 {BENCH_URL} {BENCH_DIR}")
else:
    print("Text2KGBench já existe.")

print("Clones OK.")

In [ ]:
# Célula 3 — Instalar dependências
run("pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy")
run("pip install -q statsmodels pandas")

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
print("Dependências OK.")

In [ ]:
# Célula 4 — Subir vLLM em segundo plano
# dtype=half necessário para T4 (float16). tensor-parallel-size=2 usa ambas as GPUs.

VLLM_LOG = f"{WORK_DIR}/vllm.log"
MODELO   = "Qwen/Qwen2.5-7B-Instruct-AWQ"
PORT     = 8000

vllm_cmd = (
    f"vllm serve {MODELO} "
    f"--port {PORT} "
    f"--dtype half "
    f"--tensor-parallel-size 2 "
    f"--max-model-len 4096 "
    f"--gpu-memory-utilization 0.90 "
    f"> {VLLM_LOG} 2>&1 &"
)
os.system(vllm_cmd)
print(f"vLLM iniciado. Log: {VLLM_LOG}")
print("Aguarde ~90s para o modelo carregar antes da próxima célula.")

In [ ]:
# Célula 5 — Aguardar vLLM pronto (polling /health)
import time, urllib.request, urllib.error

VLLM_URL = f"http://localhost:{PORT}"
max_wait = 300  # segundos
intervalo = 10
elapsed = 0

while elapsed < max_wait:
    try:
        urllib.request.urlopen(f"{VLLM_URL}/health", timeout=3)
        print(f"vLLM pronto após {elapsed}s.")
        break
    except (urllib.error.URLError, ConnectionRefusedError):
        print(f"  aguardando... ({elapsed}s)")
        time.sleep(intervalo)
        elapsed += intervalo
else:
    print("TIMEOUT: vLLM não respondeu. Verifique o log:")
    run(f"tail -50 {VLLM_LOG}")

In [ ]:
# Célula 6 — Gerar manifesto do piloto com caminhos do Kaggle
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)

run(
    f"python {REPO_DIR}/scripts/gerar_manifesto.py "
    f"--url http://localhost:{PORT}/v1 "
    f"--saida_dir {REPO_DIR}/configs/pipeline",
    cwd=REPO_DIR
)

# Lê o manifesto piloto e substitui caminhos de saída para o Kaggle
import json
piloto_path = f"{REPO_DIR}/configs/pipeline/manifesto_piloto_kaggle.json"
with open(piloto_path, encoding="utf-8") as f:
    piloto = json.load(f)

for exp in piloto:
    exp["saida"] = f"{RESULT_DIR}/{os.path.basename(exp['saida'])}"

piloto_kaggle_path = f"{WORK_DIR}/manifesto_piloto_kaggle.json"
with open(piloto_kaggle_path, "w", encoding="utf-8") as f:
    json.dump(piloto, f, ensure_ascii=False, indent=2)

print(f"Manifesto com {len(piloto)} experimentos -> {piloto_kaggle_path}")

In [ ]:
# Célula 7 — Executar piloto (C0/C1/C1_oneshot, 2 ontologias, 200 frases)
# --skip_existentes permite retomar se a sessão foi interrompida

run(
    f"python -m t2kg.run "
    f"--manifesto {piloto_kaggle_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--skip_existentes "
    f"--verbose",
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
)
print("\nPiloto concluído.")

In [ ]:
# Célula 8 — Indução C2: extração livre nas 50 frases de treino (D-16)
# Roda DEPOIS do piloto para reutilizar o vLLM já carregado.
# Produz configs/c2/{dataset}_{onto}_induced.json e dados de calibração.

MODELO_LLM = "Qwen/Qwen2.5-7B-Instruct-AWQ"

for onto in ["1_movie", "2_music"]:
    print(f"\n--- Induzindo C2 para tekgen/{onto} ---")
    run(
        f"python {REPO_DIR}/scripts/induzir_c2.py "
        f"--dataset tekgen "
        f"--onto {onto} "
        f"--url http://localhost:{PORT}/v1 "
        f"--modelo_llm {MODELO_LLM} "
        f"--threshold 0.80 "
        f"--n_frases 50 "
        f"--seed 42 "
        f"--calibrar "
        f"--saida_dir {C2_DIR}",
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
    )

print("\nIndução C2 concluída.")
print(f"Arquivos em: {C2_DIR}")

In [ ]:
# Célula 9 — Avaliar resultados do piloto

CSV_OUT = f"{RESULT_DIR}/tabela_piloto.csv"

run(
    f"python {REPO_DIR}/scripts/avaliar_resultados.py "
    f"--manifesto {piloto_kaggle_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--csv {CSV_OUT}",
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}
)

import pandas as pd
df = pd.read_csv(CSV_OUT)
print(df.to_string())

In [ ]:
# Célula 10 — Salvar resultados no repositório e fazer commit
# ATENÇÃO: isso faz push para o repositório privado usando GITHUB_TOKEN.
# Execute apenas se quiser persistir os resultados no git.

GIT_EMAIL = "felipemoraissoares@gmail.com"
GIT_NAME  = "Felipe Morais (Kaggle)"

run(f"git -C {REPO_DIR} config user.email '{GIT_EMAIL}'")
run(f"git -C {REPO_DIR} config user.name '{GIT_NAME}'")
run(f"git -C {REPO_DIR} add resultados/piloto/ configs/c2/")

commit_msg = "resultados/piloto: Qwen2.5-7B, 200 frases, C0/C1/C1_oneshot + C2 inducao"
run(f'git -C {REPO_DIR} commit -m "{commit_msg}" || echo "Nada para commitar"')
run(f"git -C {REPO_DIR} push")
print("Push concluído.")

In [ ]:
# Célula 11 — Verificar log do vLLM (diagnóstico)
run(f"tail -30 {VLLM_LOG}")

## Retomada de sessão interrompida

Se a sessão Kaggle foi interrompida antes do fim, retome a partir da **Célula 5** 
(aguardar vLLM). O vLLM precisa ser reiniciado (Célula 4) pois o processo não persiste.
As células 7 e 8 usam `--skip_existentes` / verificam arquivos existentes, então os
experimentos já concluídos são pulados automaticamente.

Para continuar a partir de um ponto específico: execute as células 1-5 (setup) e então
a partir da célula onde a sessão parou.